# Module 06 — One-way ANOVA, Tukey Post Hoc, Two-way ANOVA
Dedicated notebook for Module 06 — reproduces one-way ANOVA + Tukey HSD (PSPP can do this, see `06_anova.sps`), PLUS two-way ANOVA + interaction effect (PSPP 2.1.1 does NOT support `UNIANOVA` — only runs here via `statsmodels`).

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

SPSS equivalents: `Analyze > Compare Means > One-Way ANOVA` (+ Post Hoc Tukey), `Analyze > General Linear Model > Univariate` (two-way).

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

## 1. One-way ANOVA: Math score by school

🎯 **What question does this method answer?** One-way ANOVA tests: do the means of 3+ groups have AT LEAST ONE pair that differs significantly — null hypothesis (H0): ALL groups share the same population mean. Important: a significant ANOVA only says “there's a difference SOMEWHERE”, it does NOT say which pair differs — that's the job of the Tukey post hoc test (section 2 below).

In [ ]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
lev_stat, lev_p = stats.levene(*groups)
print(f"Levene: F={lev_stat:.3f}, p={lev_p:.3f}")
for name, g in df.groupby("school"):
    print(f"  School {name}: n={len(g)}, M={g['math'].mean():.2f}")
print(f"ANOVA: F(2,{len(df)-3})={f_stat:.3f}, p={p_val:.4f}")

#### 📤 Actual output
School A (n=80) M≈50.67; B (n=99) M≈47.26; C (n=61) M≈45.34. F(2,237)=5.87, p=0.003 → a significant difference. Levene p=0.292 (≥0.05, variances are homogeneous, so Tukey is appropriate).

## 2. Tukey HSD post hoc — finding EXACTLY which pair differs

🎯 **What question does this method answer?** Tukey HSD tests EACH PAIR of groups individually, with an adjustment so the Type I error rate isn't “inflated” by testing many pairs at once (the multiple-comparisons problem — the more pairs you compare, the more likely you'll stumble on one “significant” pair purely by chance). This answers the question the ANOVA in section 1 left open: EXACTLY where the difference lies.

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(df["math"], df["school"], alpha=0.05)
print(tukey)

#### 📤 Actual output
A–B gap 3.41 (p≈0.046, significant); A–C gap 5.34 (p≈0.003, significant); B–C gap 1.93 (p≈0.427, not significant). Homogeneous subsets: {A} stands alone, {B, C} form one group.

## 3. Two-way ANOVA: Math score ~ school × gender (+ interaction effect)
⚠️ PSPP 2.1.1 does NOT support the `UNIANOVA` command (verified — it errors with "UNIANOVA is not yet implemented") — this part ONLY runs in the notebook, via `statsmodels.formula.api.ols` + `anova_lm`.

🎯 **What question does this method answer?** Two-way ANOVA simultaneously tests 3 questions: (1) does variable A (school) have a main effect, (2) does variable B (gender) have a main effect, (3) does the EFFECT of A DEPEND on the level of B (interaction effect). Question 3 CANNOT be answered by running two separate one-way ANOVAs — this is exactly why two-way ANOVA is needed.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

model = smf.ols("math ~ C(school) * C(gender)", data=df).fit()
aov = anova_lm(model, typ=2)
print(aov.round(4))

#### 📤 Actual output
School (main effect): F≈5.81, p≈0.003 → significant. Gender (main effect): F≈1.71, p≈0.192 → not significant. School×Gender (interaction): F≈1.16, p≈0.315 → not significant. ✅ Matches the lesson — school creates a real difference, but that difference does NOT vary meaningfully between male and female students (no interaction).

## 4. Visualising the interaction (near-parallel lines = no strong interaction)

In [ ]:
fig, ax = plt.subplots()
sns.pointplot(data=df, x="school", y="math", hue="gender", order=["A","B","C"], ax=ax, errorbar=None)
ax.set_title("Mean Math score by school × gender")
plt.show()